# RAFT-Net full runs v3 — final candidate without the full-resolution stem

Same protocol and code as `raftnet-fullrun-v2`, plus the variant `raftnet_lite_nofullres`: boundary head + 32-channel forensic pathway, SRM/Bayar residuals at 1/4 resolution, no FCSS, no ACR.

**Inputs:** CASIA v2 only. **Settings:** GPU T4 ×2, Internet on. ~12 h (short resume may be needed).

Account 1: seeds 1221 + 1222 (default `JOBS`). Account 2: replace `JOBS` with seed 1223 only.

Trains the **finalised RAFT-Net** and the **official ForMa** baseline under one identical protocol
(ImageNet VMamba-tiny, 512px, the saved 11,225/1,247 split, the same augmentation, ForMa's Dice+Focal
loss, AdamW 1e-4, 500-step warm-up + cosine over **40 epochs**, batch 8, checkpoint chosen by CASIA v2
**validation** per-image F1 — test sets are never touched here).

RAFT-Net = full-resolution forensic stem (SRM + Bayar full-res, Noiseprint++ frozen at 1/4) + **FCSS** +
widened decoder + boundary head + **ACR** (off for epochs 1–20, ramps to λ=0.05 over epochs 21–24;
fp32 on a 2-image sub-batch — the fix verified by the diagnostic and the pilot smoke test).

**How a session works.** Each job runs as its own process on its own GPU, writes its log to
`run/logs/`, and saves its complete training state every epoch. Near the 12-hour limit it pauses
cleanly. **To continue:** Save Version again with **this notebook's previous version output attached**
(Add Input → Notebook Output). Jobs resume exactly (same data order, optimizer state intact); jobs already
finished are skipped and their results carried forward, so the latest version's output always holds
everything.

**Safety:** step-0 self-check per job; skipped-step and weight-movement check every epoch — a job stops
itself after two bad epochs instead of silently wasting the session.

**Inputs:** CASIA v2 (divg07) [+ this notebook's previous output when resuming].
**Settings:** GPU **T4 ×2**, Internet on.


## 1. Setup

In [ ]:
import subprocess, sys, os

REPO_DIR = '/kaggle/working/ForMa_repo'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                     'https://github.com/multimediaFor/ForMa.git', REPO_DIR], check=True)

MODELS_DIR = os.path.join(REPO_DIR, 'models')
print('Repo cloned at:', REPO_DIR)
print('Models dir:', MODELS_DIR, '| exists:', os.path.isdir(MODELS_DIR))


# -- Fix: ForMa's own csm_triton.py has an unguarded @triton.jit decorator --------------
try:
    import triton
    print("triton already available:", triton.__version__)
except ImportError:
    print("triton not installed -- attempting pip install...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'triton'],
                    capture_output=True, text=True)
    try:
        import triton
        print("triton installed successfully:", triton.__version__)
    except ImportError:
        print("triton still unavailable -- injecting a no-op stub so csm_triton.py can import "
              "without crashing. Safe: WITH_TRITON correctly evaluates False in that file, so "
              "the actual computation already uses the verified pure-PyTorch scan regardless "
              "of this stub.")
        import types
        _triton_stub = types.ModuleType('triton')
        _triton_stub.jit = lambda fn=None, **kwargs: (fn if fn is not None else (lambda f: f))
        _tl_stub = types.ModuleType('triton.language')
        _tl_stub.constexpr = object
        _triton_stub.language = _tl_stub
        sys.modules['triton'] = _triton_stub
        sys.modules['triton.language'] = _tl_stub


In [ ]:
import importlib, subprocess, sys

def pip_install(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

for pkg, imp in [('timm', 'timm'), ('fvcore', 'fvcore'), ('einops', 'einops'), ('opencv-python-headless', 'cv2')]:
    try:
        importlib.import_module(imp)
    except ImportError:
        pip_install(pkg)

# mamba-ssm / causal-conv1d: ForMa's own code (csms6s.py) gracefully falls back to a
# pure-PyTorch selective-scan if no CUDA kernel is importable -- a failed install here
# costs speed, not correctness.
try:
    import selective_scan_cuda  # noqa: F401
    print("mamba-ssm's selective_scan_cuda already importable.")
except ImportError:
    print("Installing mamba-ssm + causal-conv1d (compiles CUDA kernels, can take a few minutes)...")
    try:
        pip_install('causal-conv1d>=1.2.0', 'mamba-ssm', '--no-build-isolation')
        import selective_scan_cuda  # noqa: F401
        print("mamba-ssm installed and selective_scan_cuda is importable.")
    except Exception as e:
        print("mamba-ssm install/import did not succeed:", e)
        print("Not fatal for THIS model -- falls back to a pure-PyTorch scan (slower, correct).")


In [ ]:
vmamba_path = os.path.join(MODELS_DIR, 'vmamba_pixelshuf_modals.py')
with open(vmamba_path, 'r') as f:
    src = f.read()

patch1_old = '''with open('../config_pixelshuf_modals.yaml', 'r') as f:
    args = yaml.safe_load(f)
device = get_device(args["cuda_idx"])'''
patch1_new = '''args = {"cuda_idx": 0}  # only args["cuda_idx"] is referenced elsewhere in this file
device = get_device(0)'''

patch2_old = '''        with torch.cuda.device(args["cuda_idx"]):
            y = self.forward_core(x)'''
patch2_new = '''        if torch.cuda.is_available():
            with torch.cuda.device(args["cuda_idx"]):
                y = self.forward_core(x)
        else:
            y = self.forward_core(x)'''

# Patch 3: SS2D's default forward_type "v2" hardcodes selective_scan_backend="oflex" --
# VMamba's own separate custom CUDA kernel, never installed here. mamba-ssm's kernel
# (backend="mamba") DID import successfully -- swapping the string is a pure bugfix.
patch3_old = '''            v2=partial(self.forward_corev2, force_fp32=(not self.disable_force32), selective_scan_backend="oflex"),#core
            v3=partial(self.forward_corev2, force_fp32=False, selective_scan_backend="oflex"),'''
patch3_new = '''            v2=partial(self.forward_corev2, force_fp32=(not self.disable_force32), selective_scan_backend="mamba"),#core -- patched: oflex kernel unavailable, mamba backend verified working
            v3=partial(self.forward_corev2, force_fp32=False, selective_scan_backend="mamba"),  # patched, same reason'''

# Patch 5: a leftover debug print fires on every forward pass, flooding the log.
patch5_old = '''        x = self.patch_embed(x)#[4, 128, 128, 96]
        print(x.shape,"=================")'''
patch5_new = '''        x = self.patch_embed(x)#[4, 128, 128, 96]'''

n_patches = 0
if patch1_old in src:
    src = src.replace(patch1_old, patch1_new); n_patches += 1
if patch2_old in src:
    src = src.replace(patch2_old, patch2_new); n_patches += 1
if patch3_old in src:
    src = src.replace(patch3_old, patch3_new); n_patches += 1
if patch5_old in src:
    src = src.replace(patch5_old, patch5_new); n_patches += 1

with open(vmamba_path, 'w') as f:
    f.write(src)

print(f"Applied {n_patches}/4 patches to {vmamba_path}")
assert n_patches == 4, "Expected patch text not found -- ForMa's repo may have changed upstream."

# Patch 4: dispatch-fallback safety net in csms6s.py, defense in depth beyond patch 3.
csms6s_path = os.path.join(MODELS_DIR, 'csms6s.py')
with open(csms6s_path, 'r') as f:
    csms6s_src = f.read()

patch4_old = '''    WITH_CUDA = (WITH_SELECTIVESCAN_OFLEX or WITH_SELECTIVESCAN_CORE or WITH_SELECTIVESCAN_MAMBA)
    fn = selective_scan_torch if backend == "torch" or (not WITH_CUDA) else SelectiveScanCuda.apply
    return fn(u, delta, A, B, C, D, delta_bias, delta_softplus, oflex, backend)'''
patch4_new = '''    WITH_CUDA = (WITH_SELECTIVESCAN_OFLEX or WITH_SELECTIVESCAN_CORE or WITH_SELECTIVESCAN_MAMBA)
    _avail = {"oflex": WITH_SELECTIVESCAN_OFLEX, "core": WITH_SELECTIVESCAN_CORE, "mamba": WITH_SELECTIVESCAN_MAMBA}
    if backend == "torch" or not WITH_CUDA:
        backend = "torch"
    elif backend not in _avail or not _avail.get(backend, False):
        backend = next((b for b in ("mamba", "core", "oflex") if _avail[b]), "torch")
    fn = selective_scan_torch if backend == "torch" else SelectiveScanCuda.apply
    return fn(u, delta, A, B, C, D, delta_bias, delta_softplus, oflex, backend)'''

n_patches_4 = 0
if patch4_old in csms6s_src:
    csms6s_src = csms6s_src.replace(patch4_old, patch4_new)
    n_patches_4 = 1

with open(csms6s_path, 'w') as f:
    f.write(csms6s_src)

print(f"Applied {n_patches_4}/1 dispatch-fallback patch to {csms6s_path}")
assert n_patches_4 == 1, "Expected patch text not found in csms6s.py -- ForMa's repo may have changed upstream."


## 2. FCSS patch

In [ ]:
# ── FCSS: patch VMamba's selective scan so a per-position forensic bias can steer the step size ──
# Must run BEFORE vmamba_pixelshuf_modals is imported (each SS2D binds its scan function at construction).
# A model that never sets `forensic_bias` (the ForMa baseline) runs exactly as before.
_vm = os.path.join(MODELS_DIR, 'vmamba_pixelshuf_modals.py')
_src = open(_vm).read()
_anchor = "            xs = xs.view(B, -1, L)\n            dts = dts.contiguous().view(B, -1, L)\n"
_inject = _anchor + (
    "            _fb = getattr(self, 'forensic_bias', None)  # RAFT-Net FCSS\n"
    "            if _fb is not None:\n"
    "                if tuple(_fb.shape[-2:]) != (H, W):\n"
    "                    _fb = F.interpolate(_fb, size=(H, W), mode='bilinear', align_corners=False)\n"
    "                _fbs = cross_scan_fn(_fb.to(x.dtype).contiguous(), in_channel_first=True, out_channel_first=True,\n"
    "                                     scans=_scan_mode, force_torch=scan_force_torch)\n"
    "                dts = dts + _fbs.contiguous().view(B, -1, L).to(dts.dtype)\n")
if 'RAFT-Net FCSS' in _src:
    print('FCSS patch already present')
else:
    assert _src.count(_anchor) == 1, 'FCSS anchor not found exactly once -- ForMa code changed?'
    open(_vm, 'w').write(_src.replace(_anchor, _inject))
    print('FCSS patch applied to', _vm)


## 3. Sources

In [ ]:
MODEL_SRC = r'''
# raftnet.py -- RAFT-Net and the official ForMa baseline, built on ForMa's own classes.
import torch
import torch.nn as nn
import torch.nn.functional as F
from vmamba_pixelshuf_modals import (Backbone_VSSM, SegFormerHead, ModalMixer, ModalitiesExtractor, ConvModule)

IMNET_MEAN = (0.485, 0.456, 0.406)
IMNET_STD = (0.229, 0.224, 0.225)


class Denorm(nn.Module):
    """Model input is ImageNet-normalised (for the backbone); forensic extractors need raw [0,1] pixels
    (Noiseprint++ was trained on [0,1] RGB). Applied identically in ForMa and RAFT-Net."""
    def __init__(self):
        super().__init__()
        self.register_buffer('mean', torch.tensor(IMNET_MEAN).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(IMNET_STD).view(1, 3, 1, 1))

    def forward(self, x):
        return x * self.std + self.mean


def conv_bn_relu(cin, cout, k=3, s=1):
    return nn.Sequential(nn.Conv2d(cin, cout, k, s, k // 2, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


# ── Official ForMa baseline ───────────────────────────────────────────────────────────────────
class ForMaOfficial(nn.Module):
    """Forensic_Vmamba as released: VMamba-T backbone; Noiseprint++ (frozen) + Bayar + SRM on the image
    shrunk to 1/4, mixed to 3 channels, fed to the decoder only."""
    def __init__(self, np_weights):
        super().__init__()
        self.denorm = Denorm()
        self.backbone = Backbone_VSSM()
        self.decode = SegFormerHead()
        self.modalmixer = ModalMixer()
        self.modal_extractor = ModalitiesExtractor(['noiseprint', 'bayar', 'srm'], np_weights)

    def train(self, mode=True):
        super().train(mode)
        self.modal_extractor.noiseprint.eval()     # ForMa's model.train() would silently re-enable BN updates
        return self

    def forward(self, x, dct=None, return_feat=False):
        n, _, h, w = x.shape
        y = self.backbone(x)
        xm = F.interpolate(self.denorm(x), size=(h // 4, w // 4), mode='bilinear', align_corners=False)
        xm = self.modalmixer(self.modal_extractor.multi_output(xm))
        return {'mask': self.decode(y, xm)}


class ForMaWide(ForMaOfficial):
    """Official ForMa with ONE change: the width w of its forensic pathway. ForMa's mixer compresses 96 channels of
    noise features to 3 (then linearly expands them to 16 before fusion with 384 RGB channels). Here the mixer outputs
    w channels and w channels enter the fusion layer; backbone, extractors, mixer blocks and decoder are unchanged."""
    def __init__(self, np_weights, width):
        super().__init__(np_weights)
        self.modalmixer = ModalMixer(out_channels=width)
        self.decode.conv_modals = nn.Conv2d(width, width, kernel_size=1)
        self.decode.linear_fuse = ConvModule(c1=96 * 4 + width, c2=96, k=1)


# ── RAFT-Net components ───────────────────────────────────────────────────────────────────────
class ForensicStem(nn.Module):
    """Forensic evidence at every backbone scale (1/4, 1/8, 1/16, 1/32).
    fullres=True : SRM + Bayar residuals computed on the FULL-resolution image, downsampled by learned
                   stride-2 convolutions (ForMa computes them on a bilinearly shrunk image, which blurs away
                   the fine residuals these filters exist to capture).
    Noiseprint++ : frozen camera-fingerprint net, at 1/4 as in ForMa (full resolution is ~16x too slow on a T4).
    dct=True     : native-grid block-DCT map (computed in the data loader on the original-resolution image,
                   JPEG grid intact) injected at 1/8."""
    def __init__(self, np_weights, fullres=True, dct=True, widths=(64, 128, 192, 256)):
        super().__init__()
        self.fullres, self.use_dct, self.widths = fullres, dct, widths
        self.hp = ModalitiesExtractor(['bayar', 'srm'])
        self.npp = ModalitiesExtractor(['noiseprint'], np_weights)
        self.hp_down = (nn.Sequential(conv_bn_relu(6, 32, 3, 2), conv_bn_relu(32, 48, 3, 2)) if fullres
                        else conv_bn_relu(6, 48, 3, 1))
        self.np_conv = conv_bn_relu(3, 16, 3, 1)
        self.f4 = conv_bn_relu(48 + 16, widths[0], 3, 1)
        self.d8 = conv_bn_relu(widths[0], widths[1], 3, 2)
        if dct:
            self.dct_enc = nn.Sequential(conv_bn_relu(64, widths[1], 1, 1), conv_bn_relu(widths[1], widths[1], 3, 1))
        self.d16 = conv_bn_relu(widths[1], widths[2], 3, 2)
        self.d32 = conv_bn_relu(widths[2], widths[3], 3, 2)

    def forward(self, x01, dct=None):
        h, w = x01.shape[-2:]
        q = F.interpolate(x01, size=(h // 4, w // 4), mode='bilinear', align_corners=False)
        hp = torch.cat(self.hp.multi_output(x01 if self.fullres else q), dim=1)
        f4 = self.f4(torch.cat([self.hp_down(hp), self.np_conv(self.npp.multi_output(q)[0])], dim=1))
        f8 = self.d8(f4)
        if self.use_dct:
            if dct is None:
                raise ValueError('RAFT-Net with dct=True needs the native-grid DCT map as input')
            if dct.shape[-2:] != f8.shape[-2:]:
                dct = F.interpolate(dct, size=f8.shape[-2:], mode='nearest')
            f8 = f8 + self.dct_enc(dct.to(f8.dtype))
        f16 = self.d16(f8)
        return [f4, f8, f16, self.d32(f16)]


class FCSS(nn.Module):
    """Forensic-Conditioned Selective Scan. For every VMamba block, a ZERO-initialised 1x1 conv maps the stage's
    forensic features to a bias on the SSM step size (dt) at every position, bounded by SCALE*tanh(.). The patched
    SS2D.forward_corev2 adds it (cross-scanned into all 4 scan directions) before the selective scan. Zero init =>
    the network starts exactly as the ImageNet-pretrained VMamba; training learns how far forensic evidence
    should steer the scan."""
    SCALE = 2.0

    def __init__(self, backbone, in_widths):
        super().__init__()
        convs, self._ops = [], []
        for s, layer in enumerate(backbone.layers):
            for blk in layer.blocks:
                d_inner = blk.op.dt_projs_weight.shape[1]
                c = nn.Conv2d(in_widths[s], d_inner, 1)
                nn.init.zeros_(c.weight)
                nn.init.zeros_(c.bias)
                convs.append(c)
                self._ops.append((s, blk.op))
        self.convs = nn.ModuleList(convs)

    def set(self, feats):
        for (s, op), conv in zip(self._ops, self.convs):
            op.forensic_bias = self.SCALE * torch.tanh(conv(feats[s]))

    def clear(self):
        for _, op in self._ops:
            op.forensic_bias = None


class WideHead(SegFormerHead):
    """ForMa's pixel-shuffle decoder, with the forensic pathway widened (ForMa: 3 -> 16 channels next to 384 RGB
    channels). Same computation as SegFormerHead.forward; optionally also returns the forensic features (for ACR)."""
    def __init__(self, modal_in=64, modal_out=32, embedding_dim=96):
        super().__init__(embedding_dim=embedding_dim)
        self.conv_modals = nn.Conv2d(modal_in, modal_out, kernel_size=1)
        self.linear_fuse = ConvModule(c1=embedding_dim * 4 + modal_out, c2=embedding_dim, k=1)

    def forward(self, inputs, x_modals, return_feat=False):
        c1, c2, c3, c4 = inputs
        m = self.conv_modals(x_modals)
        n = c4.shape[0]
        _c4 = F.pixel_shuffle(self.linear_c4(c4).permute(0, 2, 1).reshape(n, -1, c4.shape[2], c4.shape[3]), 8)
        _c3 = F.pixel_shuffle(self.linear_c3(c3).permute(0, 2, 1).reshape(n, -1, c3.shape[2], c3.shape[3]), 4)
        _c2 = F.pixel_shuffle(self.linear_c2(c2).permute(0, 2, 1).reshape(n, -1, c2.shape[2], c2.shape[3]), 2)
        _c1 = self.linear_c1(c1).permute(0, 2, 1).reshape(n, -1, c1.shape[2], c1.shape[3])
        out = self.linear_pred(self.dropout(self.linear_fuse(torch.cat([_c4, _c3, _c2, _c1, m], dim=1))))
        return (out, m) if return_feat else out


class BoundaryModule(nn.Module):
    """BASNet-style boundary-motivated module (from the RAFT-Net prototype, unchanged)."""
    def __init__(self, c_low1=96, c_low2=192, c_high=768, mid_channels=64):
        super().__init__()
        self.conv_f1 = nn.Sequential(nn.Conv2d(c_low1, mid_channels, 1), nn.BatchNorm2d(mid_channels), nn.ReLU(inplace=True))
        self.conv_f2 = nn.Sequential(nn.Conv2d(c_low2, mid_channels, 1), nn.BatchNorm2d(mid_channels), nn.ReLU(inplace=True))
        self.gate = nn.Sequential(nn.Conv2d(mid_channels * 2, mid_channels, 1), nn.ReLU(inplace=True),
                                  nn.Conv2d(mid_channels, mid_channels, 1))
        self.attn = nn.Sequential(nn.Conv2d(mid_channels, mid_channels, 1), nn.Sigmoid())
        self.conv_high = nn.Sequential(nn.Conv2d(c_high, mid_channels, 1), nn.BatchNorm2d(mid_channels), nn.ReLU(inplace=True))

    def forward(self, f1, f2, f_high):
        hw = f1.shape[-2:]
        F1 = self.conv_f1(f1)
        F2 = self.conv_f2(F.interpolate(f2, size=hw, mode='bilinear', align_corners=True))
        FL = self.attn(self.gate(torch.cat([F1, F2], dim=1)))
        FH = self.conv_high(F.interpolate(f_high, size=hw, mode='bilinear', align_corners=True))
        return FL * FH


class EdgeEnhance(nn.Module):
    """BASNet-style boundary-enhanced module (from the RAFT-Net prototype, unchanged)."""
    def __init__(self, channels=64):
        super().__init__()
        self.pool1 = nn.AdaptiveAvgPool2d(1)
        self.pool2 = nn.AdaptiveAvgPool2d(4)
        self.atrous = nn.Conv2d(channels, channels, kernel_size=3, padding=2, dilation=2)
        self.fuse = nn.Conv2d(channels * 2, channels, kernel_size=1)
        self.swish = nn.SiLU(inplace=True)

    def forward(self, x):
        h, w = x.shape[-2:]
        p1 = F.interpolate(self.pool1(x), size=(h, w), mode='bilinear', align_corners=True)
        p2 = F.interpolate(self.pool2(x), size=(h, w), mode='bilinear', align_corners=True)
        return x + self.swish(self.atrous(x)) * torch.sigmoid(self.fuse(torch.cat([p1, p2], dim=1)))


class BoundaryBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.bmm = BoundaryModule()
        self.bem = EdgeEnhance(64)
        self.head = nn.Sequential(conv_bn_relu(64, 32, 3, 1), nn.Conv2d(32, 1, 1))

    def forward(self, c1, c2, c4, out_hw):
        b = self.head(self.bem(self.bmm(c1, c2, c4)))
        return F.interpolate(b, size=out_hw, mode='bilinear', align_corners=True)


# ── RAFT-Net ──────────────────────────────────────────────────────────────────────────────────
class RAFTNet(nn.Module):
    def __init__(self, np_weights, fcss=True, dct=True, fullres=True, boundary=True):
        super().__init__()
        self.cfg = dict(fcss=fcss, dct=dct, fullres=fullres, boundary=boundary)
        self.denorm = Denorm()
        self.backbone = Backbone_VSSM()
        self.stem = ForensicStem(np_weights, fullres=fullres, dct=dct)
        self.fcss = FCSS(self.backbone, self.stem.widths) if fcss else None
        self.decode = WideHead(modal_in=self.stem.widths[0], modal_out=32)
        self.boundary = BoundaryBranch() if boundary else None

    def train(self, mode=True):
        super().train(mode)
        self.stem.npp.noiseprint.eval()
        return self

    def forward(self, x, dct=None, return_feat=False):
        h, w = x.shape[-2:]
        feats = self.stem(self.denorm(x), dct)
        if self.fcss is not None:
            self.fcss.set(feats)
        try:
            y = self.backbone(x)
        finally:
            if self.fcss is not None:
                self.fcss.clear()
        out = {}
        if return_feat:
            out['mask'], out['feat'] = self.decode(y, feats[0], return_feat=True)
        else:
            out['mask'] = self.decode(y, feats[0])
        if self.boundary is not None:
            out['boundary'] = self.boundary(y[0], y[1], y[3], (h, w))
        return out


VARIANTS = {
    # name                  builder kwargs (None = official ForMa)                                    needs DCT input
    'forma':               (None,                                                                     False),
    'raftnet':             (dict(fcss=True,  dct=False, fullres=True,  boundary=True),                False),  # FINAL
    'raftnet_no_fcss':     (dict(fcss=False, dct=False, fullres=True,  boundary=True),                False),
    'raftnet_no_boundary': (dict(fcss=True,  dct=False, fullres=True,  boundary=False),               False),
    'raftnet_no_fullres':  (dict(fcss=True,  dct=False, fullres=False, boundary=True),                False),
    'raftnet_lite_nofullres': (dict(fcss=False, dct=False, fullres=False, boundary=True),             False),  # final candidate
    'forma_w12':           (dict(width=12),                                                            False),  # bottleneck study
    'forma_w48':           (dict(width=48),                                                            False),  # bottleneck study
}


def build_model(variant, np_weights):
    kw, _ = VARIANTS[variant]
    if kw is None:
        return ForMaOfficial(np_weights)
    if 'width' in kw:
        return ForMaWide(np_weights, kw['width'])
    return RAFTNet(np_weights, **kw)


def load_imagenet_backbone(model, sd, prefix='backbone.'):
    """ForMa train.py logic: copy checkpoint tensors into model[prefix + name] when shapes match."""
    msd = model.state_dict()
    n = 0
    for k, v in sd.items():
        mk = prefix + k
        if mk in msd and tuple(msd[mk].shape) == tuple(v.shape):
            msd[mk] = v
            n += 1
    model.load_state_dict(msd)
    bb = [k for k in msd if k.startswith(prefix)]
    return n, len(sd), len(bb)
'''
print('raftnet.py:', len(MODEL_SRC.splitlines()), 'lines')


In [ ]:
COMMON_SRC = r'''
# common.py -- the single protocol shared by EVERY full-run job (ForMa, RAFT-Net, all ablations).
import os, math, random
import numpy as np
import cv2
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

cv2.setNumThreads(0)
EPOCHS, BATCH, LR, WD = 40, 8, 1e-4, 1e-2
WARMUP_ITERS, MIN_LR_RATIO = 500, 0.01
INPUT = 512
LAMBDA_BOUNDARY = 1.0
ACR_START_EPOCH, ACR_RAMP_EPOCHS, ACR_LAMBDA, ACR_SUBBATCH = 20, 4, 0.05, 2   # 0-based: off for epochs 1-20
NUM_WORKERS = 2                     # two jobs share the 4 CPU cores
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def acr_lambda(epoch):
    """epoch = 0-based index of the epoch about to be trained."""
    if epoch < ACR_START_EPOCH:
        return 0.0
    return ACR_LAMBDA * min(1.0, (epoch - ACR_START_EPOCH + 1) / ACR_RAMP_EPOCHS)


# ── data (identical to the pilot, without the dropped DCT stream) ────────────────────────────
def _jpeg(img, q):
    ok, enc = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, int(q)])
    return cv2.imdecode(enc, cv2.IMREAD_COLOR) if ok else img


def augment(img, mask):
    if random.random() >= 0.75:
        return img, mask
    if random.random() < 0.75:
        k = random.randrange(4)
        if k == 0:   img, mask = img[:, ::-1], mask[:, ::-1]
        elif k == 1: img, mask = img[::-1], mask[::-1]
        elif k == 2: img, mask = np.rot90(img), np.rot90(mask)
        else:        img, mask = img.transpose(1, 0, 2), mask.T
        img, mask = np.ascontiguousarray(img), np.ascontiguousarray(mask)
    if random.random() < 0.75:
        if random.random() < 0.5:
            img = _jpeg(img, random.randint(50, 95))
        else:
            h, w = img.shape[:2]
            s = random.uniform(0.5, 0.9)
            small = cv2.resize(img, (max(1, int(w * s)), max(1, int(h * s))), interpolation=cv2.INTER_AREA)
            img = cv2.resize(_jpeg(small, random.randint(50, 85)), (w, h), interpolation=cv2.INTER_LINEAR)
    if random.random() < 0.25:
        k = random.randrange(3)
        if k == 0:   img = cv2.GaussianBlur(img, (0, 0), random.uniform(0.5, 1.5))
        elif k == 1: img = cv2.medianBlur(img, 3)
        else:        img = np.clip(img.astype(np.float32) + np.random.normal(0, random.uniform(2, 8), img.shape), 0, 255).astype(np.uint8)
    return img, mask


def load_sample(ip, mp, train):
    img = cv2.imread(ip, cv2.IMREAD_COLOR)
    h, w = img.shape[:2]
    m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE) if mp else None
    m = np.zeros((h, w), np.uint8) if m is None else (m > 127).astype(np.uint8)
    if m.shape != (h, w):
        m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
    if train:
        img, m = augment(img, m)
    x = cv2.resize(img, (INPUT, INPUT))[..., ::-1].astype(np.float32) / 255.0
    x = ((x - MEAN) / STD).transpose(2, 0, 1)
    m = cv2.resize(m, (INPUT, INPUT), interpolation=cv2.INTER_NEAREST).astype(np.float32)
    return np.ascontiguousarray(x), m


class CASIASet(Dataset):
    def __init__(self, samples, train):
        self.samples, self.train = samples, train
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        x, m = load_sample(*self.samples[i], self.train)
        return torch.from_numpy(x), torch.from_numpy(m)


def _worker_init(wid):
    s = torch.initial_seed() % (2 ** 32)
    random.seed(s); np.random.seed(s)


def train_loader(samples, seed, epoch):
    """Order and augmentation are seeded per (seed, epoch): a resumed run sees exactly the batches an
    uninterrupted run would."""
    g = torch.Generator()
    g.manual_seed(seed * 1000 + epoch)
    return DataLoader(CASIASet(samples, True), batch_size=BATCH, shuffle=True, generator=g, drop_last=True,
                      num_workers=NUM_WORKERS, worker_init_fn=_worker_init, pin_memory=True)


def val_loader(samples):
    return DataLoader(CASIASet(samples, False), batch_size=8, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)


# ── losses: ForMa's official Dice + softmax-focal for EVERY model; RAFT-Net adds boundary + ACR ──
def init_forma_loss(repo_dir):
    import importlib.util as ilu
    spec = ilu.spec_from_file_location('forma_losses', os.path.join(repo_dir, 'losses.py'))
    mod = ilu.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.MyLoss()


def boundary_target(m):
    m4 = m.unsqueeze(1)
    return (F.max_pool2d(m4, 5, 1, 2) + F.max_pool2d(-m4, 5, 1, 2)).clamp(0, 1)


def boundary_loss(logits, m):
    t = boundary_target(m)
    pos = t.mean().clamp(min=1e-4)
    bce = F.binary_cross_entropy_with_logits(logits.float(), t, pos_weight=((1 - pos) / pos).clamp(max=20.0))
    p = torch.sigmoid(logits.float())
    return bce + 1 - (2 * (p * t).sum() + 1) / (p.sum() + t.sum() + 1)


def main_loss(model, x, m, forma_loss):
    with torch.amp.autocast('cuda'):
        out = model(x)
    loss = forma_loss(out['mask'].float(), m)
    parts = {'seg': float(loss.detach())}
    if 'boundary' in out:
        lb = boundary_loss(out['boundary'], m)
        loss = loss + LAMBDA_BOUNDARY * lb
        parts['boundary'] = float(lb.detach())
    return loss, parts


def saliency_second_order(logits, feat):
    score = logits[:, 1].sum()
    grads = torch.autograd.grad(score, feat, create_graph=True, retain_graph=True)[0]
    return F.relu((grads.mean(dim=(2, 3), keepdim=True) * feat).sum(dim=1, keepdim=True))


def acr_tv_loss(S, prob, threshold=0.5):
    if S.shape[-2:] != prob.shape[-2:]:
        S = F.interpolate(S, size=prob.shape[-2:], mode='bilinear', align_corners=True)
    m_hat = (prob > threshold).float()
    dh, dw = (S[:, :, 1:] - S[:, :, :-1]).abs(), (S[:, :, :, 1:] - S[:, :, :, :-1]).abs()
    mh = ((m_hat[:, :, 1:] + m_hat[:, :, :-1]) > 0).float()
    mw = ((m_hat[:, :, :, 1:] + m_hat[:, :, :, :-1]) > 0).float()
    return (((dh * mh).sum((1, 2, 3)) + (dw * mw).sum((1, 2, 3))) / m_hat.sum((1, 2, 3)).clamp(min=1.0)).mean()


def acr_subbatch_loss(model, x, n=ACR_SUBBATCH):
    """Fixed ACR (verified by the diagnostic + pilot smoke test): second-order Grad-CAM with AMP OFF on n
    images; BatchNorm uses running statistics in this pass so the tiny sub-batch cannot corrupt them."""
    was_training = model.training
    model.eval()
    try:
        with torch.amp.autocast('cuda', enabled=False):
            out = model(x[:n].float(), None, return_feat=True)
            logits = F.interpolate(out['mask'], size=x.shape[-2:], mode='bilinear', align_corners=True)
            return acr_tv_loss(saliency_second_order(logits, out['feat']), F.softmax(logits, dim=1)[:, 1:2])
    finally:
        model.train(was_training)


def make_schedule(opt, steps_per_epoch):
    total = steps_per_epoch * EPOCHS
    def f(step):
        if step < WARMUP_ITERS:
            return (step + 1) / WARMUP_ITERS
        t = min(1.0, (step - WARMUP_ITERS) / max(1, total - WARMUP_ITERS))
        return MIN_LR_RATIO + (1 - MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * t))
    return optim.lr_scheduler.LambdaLR(opt, f)


def param_sq(model):
    return sum(float(p.detach().float().pow(2).sum()) for p in model.parameters() if p.requires_grad)


@torch.no_grad()
def validate(model, loader, device, acc_cls):
    """CASIA v2 validation, per-image F1 @0.5 (checkpoint selection only -- test sets are never used here)."""
    model.eval()
    acc = acc_cls('casia2_val')
    for x, m in loader:
        with torch.amp.autocast('cuda'):
            out = model(x.to(device, non_blocking=True))['mask']
        d = F.interpolate((out[:, 1:2] - out[:, 0:1]).float(), (INPUT, INPUT), mode='bilinear', align_corners=True)
        for p, g in zip(torch.sigmoid(d)[:, 0].cpu().numpy(), m.numpy()):
            acc.add(p, g if g.any() else None, int(g.any()))
    model.train()
    return acc.summary()
'''


In [ ]:
TRAIN_SRC = r'''
# train_job.py -- trains ONE (variant, seed, acr) job for common.EPOCHS epochs, resumable across sessions.
import os, sys, json, time, argparse, shutil
ap = argparse.ArgumentParser()
for a in ('--job', '--variant', '--models_dir', '--repo_dir', '--run_dir', '--imagenet_ckpt', '--split_json'):
    ap.add_argument(a, required=True)
ap.add_argument('--seed', type=int, required=True)
ap.add_argument('--acr', type=int, required=True)
ap.add_argument('--max_hours', type=float, required=True)
args = ap.parse_args()
sys.path.insert(0, args.models_dir)
sys.path.insert(0, args.run_dir)
T0 = time.time()
J = args.job
CK = os.path.join(args.run_dir, 'ckpt')
os.makedirs(CK, exist_ok=True)


def log(*a):
    print(f"[{J}]", *a, flush=True)


def write_json(obj, path):
    tmp = path + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(obj, f, indent=1, default=float)
    os.replace(tmp, path)


def status(state, **kw):
    write_json(dict(state=state, job=J, variant=args.variant, seed=args.seed, acr=bool(args.acr), **kw),
               os.path.join(CK, f'{J}_status.json'))


def find_state(job, bases=('/kaggle/working', '/kaggle/input')):
    """Most advanced saved state for this job, from this session or an attached previous version's output."""
    best, best_ep = None, -1
    for base in bases:
        for r, _, fs in os.walk(base):
            if f'{job}_state.json' in fs and f'{job}_state.pth' in fs:
                try:
                    ep = json.load(open(os.path.join(r, f'{job}_state.json')))['epoch']
                except Exception:
                    continue
                if ep > best_ep:
                    best, best_ep = r, ep
    return best, best_ep


if __name__ == '__main__':
    import numpy as np
    import torch
    import torch.optim as optim
    import common as C
    from raftnet import build_model, load_imagenet_backbone
    from metrics_engine import MetricAccumulator

    assert not (args.acr and args.variant.startswith('forma')), 'ACR needs return_feat (RAFT-Net variants only)'
    dev = torch.device('cuda')
    torch.backends.cudnn.benchmark = True
    split = json.load(open(args.split_json))
    TRAIN = [tuple(x) for x in split['train']]
    VAL = [tuple(x) for x in split['val']]
    assert (len(TRAIN), len(VAL)) == (11225, 1247)
    forma_loss = C.init_forma_loss(args.repo_dir)
    NP = os.path.join(args.repo_dir, 'weights', 'np++.pth')

    C.set_seed(args.seed)
    model = build_model(args.variant, NP).to(dev)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = optim.AdamW(params, lr=C.LR, weight_decay=C.WD)
    steps = len(TRAIN) // C.BATCH
    sched = C.make_schedule(opt, steps)
    scaler = torch.amp.GradScaler('cuda')
    start, best, hist = 0, -1.0, []
    meta = dict(job=J, variant=args.variant, seed=args.seed, acr=bool(args.acr), epochs=C.EPOCHS, batch=C.BATCH,
                lr=C.LR, weight_decay=C.WD, input=C.INPUT, acr_start_epoch=C.ACR_START_EPOCH, acr_lambda=C.ACR_LAMBDA,
                params_m=sum(p.numel() for p in model.parameters()) / 1e6, color='rgb', normalize='imagenet')

    src_dir, src_ep = find_state(J)
    if src_dir is not None:
        st = torch.load(os.path.join(src_dir, f'{J}_state.pth'), map_location='cpu', weights_only=False)
        assert st['meta']['variant'] == args.variant and st['meta']['seed'] == args.seed and st['meta']['acr'] == bool(args.acr)
        model.load_state_dict(st['model']); opt.load_state_dict(st['optimizer'])
        sched.load_state_dict(st['scheduler']); scaler.load_state_dict(st['scaler'])
        start, best, hist = st['epoch'], st['best_f1'], st['history']
        if os.path.abspath(src_dir) != os.path.abspath(CK):          # carry best-so-far files into this session's output
            for f in (f'{J}_best.pth', f'{J}_meta.json', f'{J}_history.json'):
                if os.path.exists(os.path.join(src_dir, f)):
                    shutil.copy(os.path.join(src_dir, f), os.path.join(CK, f))
        log(f"RESUMED at epoch {start + 1}/{C.EPOCHS} from {src_dir} | best val F1 so far {best:.4f}")
        del st
    else:
        raw = torch.load(args.imagenet_ckpt, map_location='cpu', weights_only=False)
        n_loaded, n_ck, _ = load_imagenet_backbone(model, raw['model'] if 'model' in raw else raw)
        del raw
        log(f"fresh run | ImageNet tensors loaded {n_loaded}/{n_ck} | {steps} steps/epoch | params {meta['params_m']:.2f} M")
        # step-0 self-check on a real batch (no optimizer step): finite losses, gradients present
        x, m = next(iter(C.train_loader(TRAIN, args.seed, 0)))
        x, m = x.to(dev), m.to(dev)
        loss, parts = C.main_loss(model, x, m, forma_loss)
        loss.backward()
        assert torch.isfinite(loss), 'non-finite loss at step 0'
        if args.acr:
            a = C.acr_subbatch_loss(model, x)
            a.backward()
            assert torch.isfinite(a), 'non-finite ACR at step 0'
        assert sum(int(p.grad is not None) for p in params) > 0.9 * len(params), 'gradients missing'
        opt.zero_grad(set_to_none=True)
        log(f"self-check passed: loss {float(loss):.4f} " + " ".join(f"{k} {v:.4f}" for k, v in parts.items()))
        del x, m, loss

    if start >= C.EPOCHS:
        status('done', epoch=start, best_f1=best)
        log('already complete')
        sys.exit(0)
    status('running', epoch=start, best_f1=best)
    vloader = C.val_loader(VAL)
    epoch_h, bad_epochs = [], 0
    for epoch in range(start, C.EPOCHS):
        if epoch_h and (time.time() - T0) / 3600 + 1.15 * max(epoch_h[-3:]) > args.max_hours:
            status('paused', epoch=epoch, best_f1=best)
            log(f"PAUSED before epoch {epoch + 1} (session time). Re-run the notebook with this version's output attached.")
            sys.exit(0)
        lam = C.acr_lambda(epoch) if args.acr else 0.0
        t0, skipped, n, sums, acr_sum = time.time(), 0, 0, {}, 0.0
        w0 = C.param_sq(model)
        model.train()
        for x, m in C.train_loader(TRAIN, args.seed, epoch):
            x, m = x.to(dev, non_blocking=True), m.to(dev, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss, parts = C.main_loss(model, x, m, forma_loss)
            scaler.scale(loss).backward()                        # main graph freed before the ACR pass
            if lam > 0:
                a = C.acr_subbatch_loss(model, x)
                scaler.scale(lam * a).backward()
                acr_sum += float(a.detach())
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            s0 = scaler.get_scale()
            scaler.step(opt); scaler.update(); sched.step()
            skipped += int(scaler.get_scale() < s0)
            n += 1
            for k, v in parts.items():
                sums[k] = sums.get(k, 0.0) + v
        v = C.validate(model, vloader, dev, MetricAccumulator)
        epoch_h.append((time.time() - t0) / 3600)
        rec = dict(epoch=epoch + 1, hours=epoch_h[-1], lr=opt.param_groups[0]['lr'], acr_lambda=lam,
                   acr=acr_sum / max(n, 1), skipped=skipped, weight_change=abs(C.param_sq(model) - w0),
                   val_f1=v['img_mean_f1'], val_iou=v['img_mean_iou'], val_auc=v.get('pixel_roc_auc', float('nan')),
                   **{k: s / max(n, 1) for k, s in sums.items()})
        hist.append(rec)
        improved = rec['val_f1'] > best
        if improved:
            best = rec['val_f1']
            torch.save(model.state_dict(), os.path.join(CK, f'{J}_best.pth.tmp'))
            os.replace(os.path.join(CK, f'{J}_best.pth.tmp'), os.path.join(CK, f'{J}_best.pth'))
            write_json({**meta, 'best_epoch': epoch + 1, 'best_val': rec}, os.path.join(CK, f'{J}_meta.json'))
        torch.save({'model': model.state_dict(), 'optimizer': opt.state_dict(), 'scheduler': sched.state_dict(),
                    'scaler': scaler.state_dict(), 'epoch': epoch + 1, 'best_f1': best, 'history': hist, 'meta': meta},
                   os.path.join(CK, f'{J}_state.pth.tmp'))
        os.replace(os.path.join(CK, f'{J}_state.pth.tmp'), os.path.join(CK, f'{J}_state.pth'))
        write_json({'epoch': epoch + 1, 'best_f1': best}, os.path.join(CK, f'{J}_state.json'))
        write_json(hist, os.path.join(CK, f'{J}_history.json'))
        status('running', epoch=epoch + 1, best_f1=best)
        log(f"epoch {epoch + 1}/{C.EPOCHS} | " + " ".join(f"{k} {rec[k]:.4f}" for k in sums) +
            (f" | acr {rec['acr']:.4f} (lambda {lam:.4f})" if lam > 0 else "") +
            f" | val F1 {rec['val_f1']:.4f} IoU {rec['val_iou']:.4f} AUC {rec['val_auc']:.4f} | best {best:.4f}"
            f"{' *' if improved else ''} | skipped {skipped}/{n} | lr {rec['lr']:.2e} | {60 * epoch_h[-1]:.1f} min")
        bad_epochs = bad_epochs + 1 if (skipped > 0.5 * n or rec['weight_change'] == 0.0) else 0
        if bad_epochs >= 2:
            status('failed', epoch=epoch + 1, best_f1=best, reason='frozen: most steps skipped or weights not moving')
            log("STOPPED: two consecutive epochs with most optimizer steps skipped or no weight movement. Report this log.")
            sys.exit(3)
    status('done', epoch=C.EPOCHS, best_f1=best)
    log(f"DONE | best val F1 {best:.4f} (epoch {max(hist, key=lambda h: h['val_f1'])['epoch']})")
'''


In [ ]:
METRICS_SRC = r'''# ── Unified metric engine ─────────────────────────────────────────────────────────
# Pure numpy, histogram-based: every pixel's tampered-probability is binned into 1000 bins, so
# counts at ANY threshold k/1000 are exact cumulative sums -- no pixel subsampling, bounded
# memory regardless of image size, and pixel ROC-AUC / AP computed on ALL pixels.
#
# Threshold semantics: a pixel is predicted tampered at threshold t=k/1000 iff prob >= t,
# which is exactly floor(prob*1000) >= k. At k=500 this is the standard fixed 0.5 threshold.
#
# Reported per dataset (tampered images WITH a ground-truth mask only, the Protocol-1
# convention used by MPC / PSCC-Net / TruFor tables):
#   F1/IoU/P/R per-image @0.5, averaged over images   <- HEADLINE (matches MPC Table II/III)
#   F1/IoU pooled over all pixels @0.5
#   F1/IoU pooled at the dataset-best threshold       (optimistic: threshold chosen on test data)
#   F1 per-image at each image's own best threshold   (oracle upper bound, clearly labelled)
#   pixel ROC-AUC and AP (all pixels of tampered images)
# plus image-level detection AUC (max-pooled prob), using authentic images as negatives.
import numpy as np
from sklearn.metrics import roc_auc_score

N_BINS = 1000
K_05 = 500  # bin index for threshold 0.5


def _safe_div(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    out = np.zeros(np.broadcast(a, b).shape, dtype=np.float64)
    np.divide(a, b, out=out, where=b > 0)
    return out


class MetricAccumulator:
    def __init__(self, name):
        self.name = name
        self.hist_pos = np.zeros(N_BINS, dtype=np.int64)   # GT-tampered pixels, all tampered images
        self.hist_neg = np.zeros(N_BINS, dtype=np.int64)   # GT-authentic pixels of tampered images
        self.rows = []            # per tampered image with a usable mask
        self.img_labels = []      # image-level: 1 tampered, 0 authentic
        self.img_scores = []      # image-level score: max prob
        self.n_empty_gt = 0       # "tampered" images whose mask has zero tampered pixels
        self.n_no_mask = 0        # tampered images with no mask file (image-level only)

    def add(self, prob, gt, label, key=None):
        """prob: HxW float in [0,1] at the SAME resolution as gt.
        gt: HxW {0,1} array, or None (authentic image, or tampered image without a mask).
        label: 1 tampered / 0 authentic (image level)."""
        prob = np.asarray(prob, dtype=np.float32)
        self.img_labels.append(int(label))
        self.img_scores.append(float(prob.max()))
        if gt is None:
            if label == 1:
                self.n_no_mask += 1
            return
        g = np.asarray(gt).astype(bool)
        assert g.shape == prob.shape, f"shape mismatch prob {prob.shape} vs gt {g.shape}"
        n_pos = int(g.sum())
        if n_pos == 0:
            self.n_empty_gt += 1
            return
        idx = np.minimum((prob * N_BINS).astype(np.int64), N_BINS - 1)
        hp = np.bincount(idx[g], minlength=N_BINS)
        hn = np.bincount(idx[~g], minlength=N_BINS)
        self.hist_pos += hp
        self.hist_neg += hn
        tp = np.cumsum(hp[::-1])[::-1]          # tp[k]: GT-pos pixels with prob >= k/1000
        fp = np.cumsum(hn[::-1])[::-1]
        fn = n_pos - tp
        f1_all = _safe_div(2 * tp, 2 * tp + fp + fn)
        TP, FP, FN = int(tp[K_05]), int(fp[K_05]), int(fn[K_05])
        self.rows.append({
            'key': key,
            'f1': float(_safe_div(2 * TP, 2 * TP + FP + FN)),
            'iou': float(_safe_div(TP, TP + FP + FN)),
            'precision': float(_safe_div(TP, TP + FP)),
            'recall': float(_safe_div(TP, TP + FN)),
            'f1_best_own_thr': float(f1_all.max()),
            'gt_frac': n_pos / g.size,
            'f1_all_tampered': 2 * n_pos / (2 * n_pos + (g.size - n_pos)),   # trivial baseline: every pixel predicted tampered
            'pred_frac': (TP + FP) / g.size,
            'tp': TP, 'fp': FP, 'fn': FN,
        })

    def summary(self):
        s = {'dataset': self.name, 'n_images_total': len(self.img_labels),
             'n_tampered_scored': len(self.rows), 'n_empty_gt_skipped': self.n_empty_gt,
             'n_tampered_without_mask': self.n_no_mask,
             'n_authentic': int(sum(1 for l in self.img_labels if l == 0))}
        if self.rows:
            for k in ['f1', 'iou', 'precision', 'recall', 'f1_best_own_thr', 'gt_frac', 'pred_frac', 'f1_all_tampered']:
                s[f'img_mean_{k}'] = float(np.mean([r[k] for r in self.rows]))
            TP = sum(r['tp'] for r in self.rows)
            FP = sum(r['fp'] for r in self.rows)
            FN = sum(r['fn'] for r in self.rows)
            s['pooled_f1@0.5'] = float(_safe_div(2 * TP, 2 * TP + FP + FN))
            s['pooled_iou@0.5'] = float(_safe_div(TP, TP + FP + FN))
            tp = np.cumsum(self.hist_pos[::-1])[::-1]
            fp = np.cumsum(self.hist_neg[::-1])[::-1]
            fn = self.hist_pos.sum() - tp
            f1 = _safe_div(2 * tp, 2 * tp + fp + fn)
            kb = int(np.argmax(f1))
            s['pooled_best_thr'] = kb / N_BINS
            s['pooled_f1@best'] = float(f1[kb])
            s['pooled_iou@best'] = float(_safe_div(tp[kb], tp[kb] + fp[kb] + fn[kb]))
            n_pos, n_neg = self.hist_pos.sum(), self.hist_neg.sum()
            if n_pos > 0 and n_neg > 0:
                # ROC-AUC = P(score_pos > score_neg) + 0.5 * P(tie within a bin)
                neg_below = np.concatenate([[0], np.cumsum(self.hist_neg)[:-1]])
                s['pixel_roc_auc'] = float((self.hist_pos * (neg_below + 0.5 * self.hist_neg)).sum()
                                           / (float(n_pos) * float(n_neg)))
                # AP = sum over thresholds (high -> low) of precision * delta-recall
                prec = _safe_div(tp, tp + fp)
                rec = tp / n_pos
                rec_next = np.concatenate([rec[1:], [0.0]])
                s['pixel_ap'] = float(((rec - rec_next) * prec).sum())
        labels = np.array(self.img_labels)
        if len(set(labels.tolist())) == 2:
            s['image_auc_maxpool'] = float(roc_auc_score(labels, np.array(self.img_scores)))
        return s
'''


In [ ]:
# ── Dataset loaders: every entry is (image_path, mask_path_or_None, label, mask_kind) ──────
# mask_kind tells load_gt() how to turn the mask file into {0,1}:
#   'binary'   -> grayscale > 127 (CASIA v1/v2, IMD2020)
#   'columbia' -> colour-coded region map: tampered = pixels where the COLUMBIA_TAMPERED
#                 colour channel dominates (NOT grayscale>127, which keeps only the thin
#                 bright-green boundary band -- the bug behind the old Columbia numbers)
import os, glob, json, random
import numpy as np
import cv2

IMG_EXTS = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp')
COLUMBIA_TAMPERED = 'green'   # verify against the README printed in the Columbia cell


def load_gt(mask_path, mask_kind, target_hw):
    """Returns a {0,1} uint8 mask at target_hw=(H, W) of the ORIGINAL image, or None."""
    if mask_path is None:
        return None
    if mask_kind == 'columbia':
        bgr = cv2.imread(mask_path, cv2.IMREAD_COLOR)
        if bgr is None:
            return None
        g = bgr[..., 1].astype(np.int16)
        r = bgr[..., 2].astype(np.int16)
        m = (g > r) if COLUMBIA_TAMPERED == 'green' else (r > g)
        m = m.astype(np.uint8)
    else:
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            return None
        m = (m > 127).astype(np.uint8)
    if m.shape[:2] != tuple(target_hw):
        m = cv2.resize(m, (target_hw[1], target_hw[0]), interpolation=cv2.INTER_NEAREST)
    return m


# ── CASIA v2 in-distribution split ─────────────────────────────────────────────────────
# 'validated' = the split used by every RAFT-Net stage, MPC and LoMa (unreadable files such as
#               Thumbs.db dropped BEFORE shuffling -> 12472 samples -> 11225/1247).
# 'cfl'       = CFL-Net's split (nothing dropped -> 12473 samples incl. Thumbs.db -> 11226/1247).
# The two share only ~41% of their validation images, so CFL-Net must be scored on its OWN
# split (the other one contains ~59% of CFL-Net's training images).
def casia2_file_list(au_dir, tp_dir, gt_dir):
    samples = []
    for f in sorted(os.listdir(au_dir)):
        p = os.path.join(au_dir, f)
        if os.path.isfile(p) and not f.startswith('.'):
            samples.append((p, None))
    gt_by_stem = {}
    for f in os.listdir(gt_dir):
        if f.startswith('.'):
            continue
        stem = os.path.splitext(f)[0]
        if stem.endswith('_gt'):
            stem = stem[:-3]
        gt_by_stem[stem] = os.path.join(gt_dir, f)
    for f in sorted(os.listdir(tp_dir)):
        p = os.path.join(tp_dir, f)
        if not (os.path.isfile(p) and not f.startswith('.')):
            continue
        m = gt_by_stem.get(os.path.splitext(f)[0])
        if m is not None:
            samples.append((p, m))
    return samples


def casia2_val_split(au_dir, tp_dir, gt_dir, variant, seed=1221, val_fraction=0.1,
                     readable_check=None):
    samples = casia2_file_list(au_dir, tp_dir, gt_dir)
    if variant == 'validated':
        assert readable_check is not None
        samples = [s for s in samples if readable_check(s)]
    elif variant != 'cfl':
        raise ValueError(variant)
    random.Random(seed).shuffle(samples)
    n_val = int(len(samples) * val_fraction)
    val, train = samples[:n_val], samples[n_val:]
    # CFL's list can contain an unreadable file in val; it is skipped at load time exactly as
    # CFL-Net's own evaluation did (reported as skipped).
    as_entries = lambda lst: [(img, m, 0 if m is None else 1, 'binary') for img, m in lst]
    return as_entries(val), as_entries(train)


# ── Columbia ────────────────────────────────────────────────────────────────────────────
def columbia_entries(root):
    splc_img = [d for d in glob.glob(os.path.join(root, '**', '4cam_splc'), recursive=True)
                if any(f.lower().endswith(('.tif', '.tiff')) for f in os.listdir(d))]
    auth_img = [d for d in glob.glob(os.path.join(root, '**', '4cam_auth'), recursive=True)
                if any(f.lower().endswith(('.tif', '.tiff')) for f in os.listdir(d))]
    edge_dirs = [d for d in glob.glob(os.path.join(root, '**', 'edgemask'), recursive=True)
                 if '4cam_splc' in d]
    assert splc_img and auth_img and edge_dirs, (splc_img, auth_img, edge_dirs)
    edge = {}
    for d in edge_dirs:
        for f in os.listdir(d):
            stem = os.path.splitext(f)[0]
            if stem.endswith('_edgemask'):          # '_edgemask_3' variants are ignored
                edge[stem[:-len('_edgemask')]] = os.path.join(d, f)
    entries, unmatched = [], []
    for f in sorted(os.listdir(splc_img[0])):
        if not f.lower().endswith(('.tif', '.tiff')):
            continue
        stem = os.path.splitext(f)[0]
        m = edge.get(stem)
        if m is None:
            unmatched.append(f)
        entries.append((os.path.join(splc_img[0], f), m, 1, 'columbia'))
    for f in sorted(os.listdir(auth_img[0])):
        if f.lower().endswith(('.tif', '.tiff')):
            entries.append((os.path.join(auth_img[0], f), None, 0, 'binary'))
    return entries, unmatched


# ── IMD2020 ─────────────────────────────────────────────────────────────────────────────
def imd2020_entries(root):
    auth_dir, img_dir, mask_dir = (os.path.join(root, d) for d in ('Authentic', 'img', 'mask'))
    masks = {}
    for f in os.listdir(mask_dir):
        stem = os.path.splitext(f)[0]
        if stem.lower().endswith('_mask'):
            masks[stem[:-5]] = os.path.join(mask_dir, f)
    entries, unmatched = [], []
    for f in sorted(os.listdir(img_dir)):
        if not f.lower().endswith(IMG_EXTS):
            continue
        m = masks.get(os.path.splitext(f)[0])
        if m is None:
            unmatched.append(f)
        entries.append((os.path.join(img_dir, f), m, 1, 'binary'))
    for f in sorted(os.listdir(auth_dir)):
        if f.lower().endswith(IMG_EXTS):
            entries.append((os.path.join(auth_dir, f), None, 0, 'binary'))
    return entries, unmatched


# ── CASIA v1 ────────────────────────────────────────────────────────────────────────────
# The attached image mirror ships NO masks. Masks come from a separate groundtruth dataset
# (e.g. the community "CASIA 1.0 groundtruth" set: Sp_..._gt.png). Discovery is by NAME,
# not by folder layout: any image file under /kaggle/input whose stem (minus an optional
# '_gt' suffix) equals a CASIA1 tampered-image stem is taken as that image's mask.
def casia1_entries(casia1_root, search_root='/kaggle/input'):
    sp_dir = os.path.join(casia1_root, 'Sp')
    au_dir = os.path.join(casia1_root, 'Au')
    tamp = {os.path.splitext(f)[0]: os.path.join(sp_dir, f)
            for f in sorted(os.listdir(sp_dir)) if f.lower().endswith(IMG_EXTS)}
    masks = {}
    for r, dirs, files in os.walk(search_root):
        if os.path.abspath(r).startswith(os.path.abspath(sp_dir)):
            continue
        for f in files:
            if not f.lower().endswith(IMG_EXTS):
                continue
            stem = os.path.splitext(f)[0]
            base = stem[:-3] if stem.endswith('_gt') else stem
            if base in tamp and (stem.endswith('_gt') or 'truth' in r.lower() or 'mask' in r.lower()
                                 or 'gt' in os.path.basename(r).lower()):
                masks[base] = os.path.join(r, f)
    entries = [(p, masks.get(s), 1, 'binary') for s, p in tamp.items()]
    entries += [(os.path.join(au_dir, f), None, 0, 'binary')
                for f in sorted(os.listdir(au_dir)) if f.lower().endswith(IMG_EXTS)]
    return entries, sum(1 for s in tamp if s not in masks)


## 4. Prepare (CPU only)

In [ ]:
# ── prepare the run directory (CPU only: this process never touches the GPUs) ───────────────
import os, sys, json, time, glob, shutil, subprocess
SESSION_T0 = time.time()
MAX_SESSION_HOURS = 11.5
RUN_DIR = '/kaggle/working/run'
LOG_DIR, CK_DIR = os.path.join(RUN_DIR, 'logs'), os.path.join(RUN_DIR, 'ckpt')
for d in (RUN_DIR, LOG_DIR, CK_DIR):
    os.makedirs(d, exist_ok=True)
with open(os.path.join(MODELS_DIR, 'raftnet.py'), 'w') as f:
    f.write(MODEL_SRC)
for name, src in (('common.py', COMMON_SRC), ('train_job.py', TRAIN_SRC), ('metrics_engine.py', METRICS_SRC)):
    with open(os.path.join(RUN_DIR, name), 'w') as f:
        f.write(src)
print('written:', sorted(os.listdir(RUN_DIR)), '+ raftnet.py in', MODELS_DIR)

def find_attached(name, base='/kaggle/input'):
    for r, _, fs in os.walk(base):
        if name in fs:
            return os.path.join(r, name)
    return None

CASIA2_ROOT = '/kaggle/input/datasets/divg07/casia-20-image-tampering-detection-dataset/CASIA2'
SPLIT_JSON = os.path.join(RUN_DIR, 'casia2_split_seed1221.json')
_att = find_attached('casia2_split_seed1221.json')
if _att and os.path.abspath(_att) != os.path.abspath(SPLIT_JSON):
    shutil.copy(_att, SPLIT_JSON)
    print('split copied from', _att)
elif not os.path.exists(SPLIT_JSON):
    import cv2
    from concurrent.futures import ThreadPoolExecutor
    _au, _tp, _gt = f'{CASIA2_ROOT}/Au', f'{CASIA2_ROOT}/Tp', f'{CASIA2_ROOT}/CASIA 2 Groundtruth'
    _all = casia2_file_list(_au, _tp, _gt)
    def _readable(s):
        return cv2.imread(s[0]) is not None and (s[1] is None or cv2.imread(s[1]) is not None)
    with ThreadPoolExecutor(8) as ex:
        _bad = {s for s, ok in zip(_all, ex.map(_readable, _all)) if not ok}
    _val, _train = casia2_val_split(_au, _tp, _gt, 'validated', readable_check=lambda s: s not in _bad)
    json.dump({'seed': 1221, 'train': [[a, b] for a, b, _, _ in _train], 'val': [[a, b] for a, b, _, _ in _val]},
              open(SPLIT_JSON, 'w'))
    print('split rebuilt (validated, seed 1221)')
_s = json.load(open(SPLIT_JSON))
assert (len(_s['train']), len(_s['val'])) == (11225, 1247), (len(_s['train']), len(_s['val']))
print(f"split OK: train {len(_s['train'])} | val {len(_s['val'])}")

VMAMBA_URL = 'https://github.com/MzeroMiko/VMamba/releases/download/%23v2cls/vssm1_tiny_0230s_ckpt_epoch_264.pth'
IMAGENET_CKPT = find_attached('vssm1_tiny_0230s_ckpt_epoch_264.pth') or '/tmp/pretrained/vssm1_tiny_0230s_ckpt_epoch_264.pth'
if not os.path.exists(IMAGENET_CKPT):
    os.makedirs(os.path.dirname(IMAGENET_CKPT), exist_ok=True)
    import torch
    torch.hub.download_url_to_file(VMAMBA_URL, IMAGENET_CKPT, progress=False)
print(f"ImageNet checkpoint: {IMAGENET_CKPT} ({os.path.getsize(IMAGENET_CKPT) / 1e6:.0f} MB)")
N_GPUS = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.count('GPU ')
print('GPUs:', N_GPUS)


## 5. Launch + monitor

In [ ]:
# ── launch: one job per GPU, live log monitoring ────────────────────────────────────────────
JOBS = [                                   # final candidate: RAFT-Net-lite with the forensic stem at 1/4 resolution
    dict(job='raftnet_lite_nofullres_s1221', variant='raftnet_lite_nofullres', seed=1221, acr=0),
    dict(job='raftnet_lite_nofullres_s1222', variant='raftnet_lite_nofullres', seed=1222, acr=0),
]

def prior_status(job):
    """most advanced status for this job in this session or an attached previous output"""
    best = None
    for base in ('/kaggle/working', '/kaggle/input'):
        for r, _, fs in os.walk(base):
            if f'{job}_status.json' in fs:
                try:
                    s = json.load(open(os.path.join(r, f'{job}_status.json')))
                except Exception:
                    continue
                if best is None or s.get('epoch', 0) > best[0].get('epoch', 0) or s.get('state') == 'done':
                    best = (s, r)
    return best

assert N_GPUS >= 1, 'no GPU visible'
procs = []
for gpu, J in enumerate(JOBS[:N_GPUS]):
    ps = prior_status(J['job'])
    if ps and ps[0].get('state') == 'done':
        src = ps[1]
        for f in os.listdir(src):                                  # carry finished results forward
            if f.startswith(J['job'] + '_') and not f.endswith('_state.pth') and os.path.abspath(src) != os.path.abspath(CK_DIR):
                shutil.copy(os.path.join(src, f), os.path.join(CK_DIR, f))
        print(f"{J['job']}: already DONE (best val F1 {ps[0].get('best_f1', float('nan')):.4f}) -- results carried forward, GPU {gpu} idle")
        continue
    budget = MAX_SESSION_HOURS - (time.time() - SESSION_T0) / 3600 - 0.25
    cmd = [sys.executable, os.path.join(RUN_DIR, 'train_job.py'), '--job', J['job'], '--variant', J['variant'],
           '--seed', str(J['seed']), '--acr', str(J['acr']), '--max_hours', f'{budget:.2f}',
           '--models_dir', MODELS_DIR, '--repo_dir', REPO_DIR, '--run_dir', RUN_DIR,
           '--imagenet_ckpt', IMAGENET_CKPT, '--split_json', SPLIT_JSON]
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED='1')
    logf = open(os.path.join(LOG_DIR, f"{J['job']}.log"), 'a')
    procs.append((J, subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=env), logf))
    print(f"launched {J['job']} on GPU {gpu} (budget {budget:.1f} h)")
if len(JOBS) > N_GPUS:
    print(f"!!! {len(JOBS) - N_GPUS} job(s) not launched: only {N_GPUS} GPU(s) visible")

offsets, last_beat = {}, time.time()
while any(p.poll() is None for _, p, _ in procs):
    time.sleep(60)
    for J, p, _ in procs:
        path = os.path.join(LOG_DIR, f"{J['job']}.log")
        with open(path, errors='replace') as f:
            f.seek(offsets.get(path, 0))
            new = f.read()
            offsets[path] = f.tell()
        for line in new.splitlines():
            if line.startswith('[') or 'Error' in line or 'Traceback' in line or 'error' in line.lower():
                print(line)
    if time.time() - last_beat > 1800:
        print(f"... {(time.time() - SESSION_T0) / 3600:.1f} h into the session | running: "
              + ", ".join(J['job'] for J, p, _ in procs if p.poll() is None))
        last_beat = time.time()
    if (time.time() - SESSION_T0) / 3600 > MAX_SESSION_HOURS + 0.2:
        print('!!! session hard limit: terminating jobs (their last completed epoch is saved)')
        for _, p, _ in procs:
            if p.poll() is None:
                p.terminate()
for J, p, logf in procs:
    logf.close()
    print(f"{J['job']} exited with code {p.returncode}")


## 6. Summary

In [ ]:
# ── session summary ─────────────────────────────────────────────────────────────────────────
print("=" * 96)
print(f"{'job':<26}{'variant':<22}{'seed':>6}{'acr':>5}{'state':>10}{'epochs':>9}{'best val F1':>13}")
print("-" * 96)
todo = []
for J in JOBS:
    p = os.path.join(CK_DIR, f"{J['job']}_status.json")
    s = json.load(open(p)) if os.path.exists(p) else {'state': 'not run'}
    print(f"{J['job']:<26}{J['variant']:<22}{J['seed']:>6}{J['acr']:>5}{s.get('state', '?'):>10}"
          f"{str(s.get('epoch', '-')) + '/40':>9}{s.get('best_f1', float('nan')):>13.4f}")
    if s.get('state') == 'failed':
        print(f"   !!! {s.get('reason')} -- send run/logs/{J['job']}.log")
    if s.get('state') in ('paused', 'running', 'not run'):
        todo.append(J['job'])
print("=" * 96)
if todo:
    print("NOT FINISHED:", ", ".join(todo))
    print("-> Save Version again with THIS version's output attached (Add Input -> Notebook Output). Keep JOBS unchanged.")
else:
    print("ALL JOBS DONE. Checkpoints for the evaluator: run/ckpt/<job>_best.pth + <job>_meta.json")
